In [24]:
!pip install -q pandas torch sentence-transformers

In [25]:
import pandas as pd
from sentence_transformers import SentenceTransformer , util
import torch
import sys

In [26]:
# Step 3: Define all the functions for our CLI tool

# A global cache to hold the model and dataset so we don't reload them.
MODEL_CACHE = {}

In [27]:
def load_model_and_data():
  """
  Loads the model and dataset into a cache to make subsequent command faster
  """
  if "model" not in MODEL_CACHE:
    print("Loading NLP model and dataset for the first time...")
    try:
      MODEL_CACHE["model"] = SentenceTransformer('all-MiniLM-L6-v2')
      df = pd.read_csv('security_nlp_cli_10000.csv')
      MODEL_CACHE["descriptions"] = df ['nl_prompt'].tolist()
      MODEL_CACHE["tools"] = df['tool'].tolist()
      MODEL_CACHE["commands"] = df['cli_command_demo'].tolist()
      # Pre-compute embeddings for speed
      print("Creating embeddings for the dataset. This may take a moment...")
      MODEL_CACHE["embeddings"] = MODEL_CACHE["model"].encode(
      MODEL_CACHE["descriptions"], convert_to_tensor=True, show_progress_bar=True
      )
      print("✅ Model and data loaded.")
    except FileNotFoundError:
      print("❌ Error: 'security_nlp_cli_10000.csv' not found. Please upload it using the file pane on the left.")
      return False
  return True



In [28]:
def get_command(input_text : str) -> str:
  """
  Takes a natural language query and returns a single, safe CLI command.
  """
  # Use the cached model and data
  model = MODEL_CACHE["model"]
  description_embeddings = MODEL_CACHE["embeddings"]
  commands = MODEL_CACHE["commands"]

  #Encode input and find similarity
  input_embedding = model.encode(input_text,convert_to_tensor = True)
  cosine_scores = util.pytorch_cos_sim(input_embedding,description_embeddings)
  top_result = torch.topk(cosine_scores , k = 1)

  #Select best tool and get command
  best_tool_idx = top_result[1][0][0].item()
  generated_command = commands[best_tool_idx]

  # Safety Filter
  blocked_keywords = ['rm -rf', 'shutdown', 'reboot', 'sudo', 'mv /', 'dd if=/dev/random']
  if any(keyword in generated_command for keyword in blocked_keywords):
      return "❌ Error: A potentially unsafe command was generated and has been blocked."
  else:
      return generated_command


In [29]:
def main():
    """
    The main function to run the interactive SecCLI tool in a Colab cell.
    """
    print("==============================================")
    print("      Welcome to SecCLI 🕵️‍♂️")
    print("==============================================")

    if not load_model_and_data():
        return

    print("\nType your security task and get a command.")
    print("Type 'exit' or 'quit' to stop.")

    while True:
        try:
            user_input = input(">> ")
            if user_input.lower() in ['exit', 'quit']:
                print("👋 Goodbye!")
                break
            if not user_input:
                continue

            final_command = get_command(user_input)
            print(f"   {final_command}\n")

        except KeyboardInterrupt:
            print("\n👋 Goodbye!")
            break
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

# Step 4: Run the main function to start the CLI
main()

      Welcome to SecCLI 🕵️‍♂️
Loading NLP model and dataset for the first time...
Creating embeddings for the dataset. This may take a moment...


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

✅ Model and data loaded.

Type your security task and get a command.
Type 'exit' or 'quit' to stop.
>> scan a subnet for live hosts
   nmap -sn 10.0.182.0/24

>>  check for an open web server port on a remote host
   nmap -sn 10.1.80.0/24 --disable-arp-ping

>> metaspolit command
   msfconsole -q

>> wireshark command to start
   tshark -i eth1 -a duration:10 -f 'tcp'

>> quit
👋 Goodbye!
